In [ ]:
import os
from pathlib import Path
# Ensure CWD is repo root so relative paths and `tools.*` imports resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")


# NER with BiLSTM-CRF and BERT Models — K-Fold Cross-Validation

Adaptation of the original notebook to use **3-fold cross-validation**, ensuring that
all 866 documents (229 annotated + 637 unannotated) are evaluated via out-of-fold predictions.
Unannotated documents carry all-O labels; if the model predicts entities on them, those count as false positives.

This allows a fair comparison with the LLM models, which are evaluated on the same 866 samples.

**Models**:
1. BiLSTM-CRF
2. BERTimbau (base and large)
3. Legal-BERTimbau, Albertina, mDeBERTa

**Output**: `.pkl` checkpoint with `{'true_labels': [...], 'pred_labels': [...]}` for
all 866 documents, per model. Metrics logged to the W&B project `eduardoplima-imd/decicontas.br`.

In [1]:
import wandb

wandb.login()
WANDB_PROJECT = 'decicontas.br'
WANDB_ENTITY = 'eduardoplima-imd'

import json
import gc
import re
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from tools.dataset import get_decicontas_df

SEED = 42
N_FOLDS = 3
torch.manual_seed(SEED)
np.random.seed(SEED)

CHECKPOINT_DIR = Path('dataset/results/checkpoints').resolve()
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: eduardoplima (eduardoplima-imd) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Checkpoints: /Users/eduardo/Dev/decicontas.br/dataset/results/checkpoints


In [2]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: mps


## 2. Data Loading and Preparation

In [3]:
def load_decicontas():
    """Carrega o dataset decicontas via get_decicontas_df e converte para formato BIO."""
    df = get_decicontas_df()
    samples = []
    for _, row in df.iterrows():
        text = row['data']['text']
        annotations = row['annotations']
        results = []
        for ann in annotations:
            if isinstance(ann, dict) and 'result' in ann:
                results.extend(ann['result'])
            elif isinstance(ann, dict) and 'value' in ann:
                results.append(ann)
        spans = []
        for r in results:
            if 'value' in r and 'labels' in r['value']:
                val = r['value']
                spans.append({'start': val['start'], 'end': val['end'], 'label': val['labels'][0]})
        samples.append({'text': text, 'spans': spans})
    annotated = [s for s in samples if s['spans']]
    print(f'Total de amostras: {len(samples)}, Anotadas: {len(annotated)}, Sem anotação: {len(samples) - len(annotated)}')
    # Retornar TODAS as amostras — sem anotação ficam com spans=[] → labels all-O
    return samples

samples = load_decicontas()

Total de amostras: 866, Anotadas: 229, Sem anotação: 637


In [4]:
def simple_tokenize(text):
    tokens = []
    for m in re.finditer(r'\S+', text):
        tokens.append({'text': m.group(), 'start': m.start(), 'end': m.end()})
    return tokens

def spans_to_bio(tokens, spans):
    labels = ['O'] * len(tokens)
    for span in spans:
        s_start, s_end, s_label = span['start'], span['end'], span['label']
        first = True
        for i, tok in enumerate(tokens):
            if tok['start'] < s_end and tok['end'] > s_start:
                if first:
                    labels[i] = f'B-{s_label}'
                    first = False
                else:
                    labels[i] = f'I-{s_label}'
    return labels

bio_data = []
for s in samples:
    tokens = simple_tokenize(s['text'])
    labels = spans_to_bio(tokens, s['spans'])
    bio_data.append({
        'tokens': [t['text'] for t in tokens],
        'token_offsets': tokens,
        'labels': labels,
        'text': s['text'],
        'spans': s['spans']
    })

all_labels = [l for s in bio_data for l in s['labels']]
label_counts = Counter(all_labels)
print('Distribuição de tags BIO:')
for k, v in sorted(label_counts.items()):
    print(f'  {k}: {v}')

unique_labels = sorted(set(all_labels))
label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}
print(f'\nTotal de tags: {len(unique_labels)}')
print(f'Total de documentos anotados: {len(bio_data)}')
print(f'K-Fold com {N_FOLDS} folds')

Distribuição de tags BIO:
  B-MULTA: 202
  B-OBRIGACAO: 119
  B-RECOMENDACAO: 56
  B-RESSARCIMENTO: 62
  I-MULTA: 11020
  I-OBRIGACAO: 7769
  I-RECOMENDACAO: 2075
  I-RESSARCIMENTO: 2865
  O: 93249

Total de tags: 9
Total de documentos anotados: 866
K-Fold com 3 folds


## 3. Evaluation Functions

In [5]:
import spacy
from seqeval.metrics import classification_report as seq_classification_report
from seqeval.metrics import f1_score as seq_f1_score
from seqeval.metrics import precision_score as seq_precision_score
from seqeval.metrics import recall_score as seq_recall_score
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
from collections import defaultdict


# ── Funções auxiliares (compartilhadas com pipeline LLM) ──────────────

def _strip_bio(tag):
    """Remove prefixos B-/I- e retorna somente o rótulo da entidade."""
    if tag == 'O' or tag is None:
        return 'O'
    parts = tag.split('-', 1)
    return parts[1] if len(parts) == 2 else parts[0]


def compute_iou_score(span_a, span_b, label_a, label_b, threshold=0.5):
    """IoU entre dois spans com rótulos."""
    s_a, e_a = span_a
    s_b, e_b = span_b
    if e_a <= s_b or e_b <= s_a:
        return 0.0
    intersection = max(0, min(e_a, e_b) - max(s_a, s_b))
    union = max(e_a, e_b) - min(s_a, s_b)
    iou = intersection / union if union > 0 else 0.0
    return 1.0 if (iou >= threshold and label_a == label_b) else 0.0


def bio_to_char_spans(bio_tags, token_offsets):
    """Converte sequência BIO + offsets de caractere em lista de char spans."""
    spans = []
    current_label = None
    current_start = None
    current_end = None
    min_len = min(len(bio_tags), len(token_offsets))

    for i in range(min_len):
        tag = bio_tags[i]
        tok = token_offsets[i]

        if tag.startswith('B-'):
            if current_label is not None:
                spans.append({'start': current_start, 'end': current_end, 'labels': [current_label]})
            current_label = tag[2:]
            current_start = tok['start']
            current_end = tok['end']
        elif tag.startswith('I-') and current_label is not None and tag[2:] == current_label:
            current_end = tok['end']
        else:
            if current_label is not None:
                spans.append({'start': current_start, 'end': current_end, 'labels': [current_label]})
                current_label = None

    if current_label is not None:
        spans.append({'start': current_start, 'end': current_end, 'labels': [current_label]})

    return spans


# ── Função de métricas unificada (mesma dos LLMs, baseada em spaCy) ───

def calculate_metrics(df, iou_threshold=0.5, spacy_model="pt_core_news_sm"):
    """
    Calcula métricas token-level e span-level usando tokenização spaCy.
    Mesmo pipeline utilizado para os LLMs, garantindo comparabilidade.
    """
    nlp = spacy.load(spacy_model)
    y_true_tokens, y_pred_tokens = [], []
    label_metrics = defaultdict(lambda: {"total_gold": 0, "total_pred": 0, "matched": 0})

    for _, row in df.iterrows():
        text = row["text"]
        doc = nlp(text)
        true_bio = ['O'] * len(doc)
        pred_bio = ['O'] * len(doc)

        for ann in row.get("golden", []):
            start, end, label = ann["start"], ann["end"], ann["labels"][0]
            cs = doc.char_span(start, end, label=label, alignment_mode="expand")
            if cs:
                for j, tok in enumerate(cs):
                    true_bio[tok.i] = f"B-{label}" if j == 0 else f"I-{label}"

        for ann in row.get("pred_as_golden", []):
            start, end, label = ann["start"], ann["end"], ann["labels"][0]
            cs = doc.char_span(start, end, label=label, alignment_mode="expand")
            if cs:
                for j, tok in enumerate(cs):
                    pred_bio[tok.i] = f"B-{label}" if j == 0 else f"I-{label}"

        y_true_tokens.append([_strip_bio(t) for t in true_bio])
        y_pred_tokens.append([_strip_bio(t) for t in pred_bio])

        gold_spans = [(a['start'], a['end'], a['labels'][0]) for a in row.get('golden', [])]
        pred_spans = [(a['start'], a['end'], a['labels'][0]) for a in row.get('pred_as_golden', [])]
        for _, _, lab in gold_spans:
            label_metrics[lab]["total_gold"] += 1
        for _, _, lab in pred_spans:
            label_metrics[lab]["total_pred"] += 1

        matched_pairs = set()
        for pi, p in enumerate(pred_spans):
            for gi, g in enumerate(gold_spans):
                if (pi, gi) in matched_pairs:
                    continue
                if compute_iou_score((p[0], p[1]), (g[0], g[1]), p[2], g[2], threshold=iou_threshold) > 0:
                    label_metrics[p[2]]["matched"] += 1
                    matched_pairs.add((pi, gi))
                    break

    # ── Token-level (ignorando B/I, sem 'O') ──
    flat_true, flat_pred = [], []
    for t_seq, p_seq in zip(y_true_tokens, y_pred_tokens):
        for t, p in zip(t_seq, p_seq):
            if t != 'O' or p != 'O':
                flat_true.append(t)
                flat_pred.append(p)

    if not flat_true:
        token_prec = token_rec = token_f1 = 0.0
        per_label = {}
    else:
        labels_sorted = sorted(set(l for l in flat_true + flat_pred if l != 'O'))
        token_prec, token_rec, token_f1, _ = precision_recall_fscore_support(
            flat_true, flat_pred, labels=labels_sorted, average='micro', zero_division=0)
        prec_l, rec_l, f1_l, sup_l = precision_recall_fscore_support(
            flat_true, flat_pred, labels=labels_sorted, average=None, zero_division=0)
        per_label = {lab: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}
                     for lab, p, r, f, s in zip(labels_sorted, prec_l, rec_l, f1_l, sup_l)}

    # ── Span-level (IoU) ──
    total_gold = sum(v["total_gold"] for v in label_metrics.values())
    total_pred = sum(v["total_pred"] for v in label_metrics.values())
    total_matched = sum(v["matched"] for v in label_metrics.values())
    iou_prec = total_matched / total_pred if total_pred > 0 else 0.0
    iou_rec = total_matched / total_gold if total_gold > 0 else 0.0
    iou_f1 = (2 * iou_prec * iou_rec / (iou_prec + iou_rec)) if (iou_prec + iou_rec) > 0 else 0.0

    iou_per_label = {}
    for label, m in label_metrics.items():
        prec = m["matched"] / m["total_pred"] if m["total_pred"] > 0 else 0.0
        rec = m["matched"] / m["total_gold"] if m["total_gold"] > 0 else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        iou_per_label[label] = {"precision": prec, "recall": rec, "f1": f1,
                                 "matched": m["matched"], "total_pred": m["total_pred"], "total_gold": m["total_gold"]}

    return {
        "token_flat": {"precision": token_prec, "recall": token_rec, "f1": token_f1, "per_label": per_label},
        "iou_agg": {"precision": iou_prec, "recall": iou_rec, "f1": iou_f1},
        "iou_per_label": iou_per_label,
    }


# ── full_evaluation: agora via spaCy (recebe bio_data) ───────────────

def full_evaluation(bio_data, oof_true, oof_pred, model_name='Model'):
    """
    Avalia predições OOF usando o pipeline spaCy unificado.
    
    Args:
        bio_data: lista de dicts com 'text', 'token_offsets', 'spans'
        oof_true: lista de sequências BIO ground truth
        oof_pred: lista de sequências BIO preditas
        model_name: nome do modelo para exibição
    """
    # Converter BIO → char spans → DataFrame
    rows = []
    for i, sample in enumerate(bio_data):
        offsets = sample['token_offsets']
        golden = bio_to_char_spans(oof_true[i], offsets)
        pred_as_golden = bio_to_char_spans(oof_pred[i], offsets)
        rows.append({
            'text': sample['text'],
            'golden': golden,
            'pred_as_golden': pred_as_golden,
        })

    df = pd.DataFrame(rows)
    raw = calculate_metrics(df, iou_threshold=0.5)

    # Exibir resultados
    print(f'\n{"="*60}')
    print(f'  {model_name} (avaliação via spaCy)')
    print(f'{"="*60}')
    print(f'Token-level F1 (micro, excl. O): {raw["token_flat"]["f1"]:.4f}')
    print(f'  Precision: {raw["token_flat"]["precision"]:.4f}')
    print(f'  Recall:    {raw["token_flat"]["recall"]:.4f}')
    print(f'Span-level F1 (IoU >= 0.5):      {raw["iou_agg"]["f1"]:.4f}')
    print(f'  Precision: {raw["iou_agg"]["precision"]:.4f}')
    print(f'  Recall:    {raw["iou_agg"]["recall"]:.4f}')
    print(f'\nPer-entity Span F1:')
    for ent, vals in sorted(raw['iou_per_label'].items()):
        print(f'  {ent}: F1={vals["f1"]:.4f} P={vals["precision"]:.4f} R={vals["recall"]:.4f}'
              f' (matched={vals["matched"]}, pred={vals["total_pred"]}, gold={vals["total_gold"]})')

    # Retornar no formato esperado pelo resto do notebook
    per_entity_f1 = {k: v['f1'] for k, v in raw['iou_per_label'].items()}

    return {
        'model': model_name,
        'token_f1': raw['token_flat']['f1'],
        'token_precision': raw['token_flat']['precision'],
        'token_recall': raw['token_flat']['recall'],
        'span_f1': raw['iou_agg']['f1'],
        'span_precision': raw['iou_agg']['precision'],
        'span_recall': raw['iou_agg']['recall'],
        'per_entity_f1': per_entity_f1,
        'raw': raw,
    }


def save_and_log_checkpoint(model_name, oof_true, oof_pred, results, technique='supervised'):
    """Salva checkpoint .pkl e loga métricas no W&B."""
    ckpt = {
        'true_labels': oof_true,
        'pred_labels': oof_pred,
        'results': results,
    }
    safe_name = model_name.replace('/', '_').replace('.', '-')
    ckpt_path = CHECKPOINT_DIR / f'{safe_name}__{technique}.pkl'
    with open(ckpt_path, 'wb') as f:
        pickle.dump(ckpt, f)
    print(f'Checkpoint salvo: {ckpt_path}')

    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
               name=f'{safe_name}__{technique}',
               config={'model': model_name, 'technique': technique,
                       'n_folds': N_FOLDS, 'n_docs': len(oof_true), 'seed': SEED})

    wandb.log({
        'token_f1': results['token_f1'],
        'span_f1': results['span_f1'],
        'span_precision': results['span_precision'],
        'span_recall': results['span_recall'],
        **{f'f1_{k}': v for k, v in results.get('per_entity_f1', {}).items()},
    })

    table = wandb.Table(columns=['doc_idx', 'true_bio', 'pred_bio'])
    for i, (t, p) in enumerate(zip(oof_true, oof_pred)):
        table.add_data(i, ' '.join(t), ' '.join(p))
    wandb.log({'predictions': table})
    wandb.finish()


print('Funções de avaliação carregadas (pipeline spaCy unificado).')


Funções de avaliação carregadas (pipeline spaCy unificado).


In [6]:
def get_checkpoint_path(model_name: str, technique: str) -> Path:
    safe_name = model_name.replace("/", "_").replace(".", "-")
    return CHECKPOINT_DIR / f"{safe_name}__{technique}.pkl"

---
## 4. BiLSTM-CRF Model — K-Fold

In [7]:
from TorchCRF import CRF
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Vocabulário construído sobre TODOS os dados (não vaza labels)
word_freq = Counter()
for s in bio_data:
    for t in s['tokens']:
        word_freq[t.lower()] += 1

vocab = ['<PAD>', '<UNK>'] + [w for w, c in word_freq.most_common() if c >= 1]
word2id = {w: i for i, w in enumerate(vocab)}
print(f'Vocab size: {len(vocab)}')


class NERDataset(Dataset):
    def __init__(self, data, word2id, label2id, max_len=512):
        self.data = data
        self.word2id = word2id
        self.label2id = label2id
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        tokens = self.data[idx]['tokens'][:self.max_len]
        labels = self.data[idx]['labels'][:self.max_len]
        input_ids = [self.word2id.get(t.lower(), self.word2id['<UNK>']) for t in tokens]
        label_ids = [self.label2id[l] for l in labels]
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'length': len(input_ids)
        }


def collate_fn(batch):
    max_len = max(b['length'] for b in batch)
    input_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels = torch.zeros(len(batch), max_len, dtype=torch.long)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, b in enumerate(batch):
        l = b['length']
        input_ids[i, :l] = b['input_ids']
        labels[i, :l] = b['labels']
        mask[i, :l] = True
    return {'input_ids': input_ids, 'labels': labels, 'mask': mask}


class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim // 2,
            num_layers=2, bidirectional=True, batch_first=True, dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
    
    def forward(self, input_ids, labels=None, mask=None):
        embeds = self.dropout(self.embedding(input_ids))
        lstm_out, _ = self.lstm(embeds)
        emissions = self.fc(self.dropout(lstm_out))
        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LABELS = len(unique_labels)
print(f'BiLSTM-CRF definido: embed={EMBEDDING_DIM}, hidden={HIDDEN_DIM}, labels={NUM_LABELS}')

bilstm_ckpt = CHECKPOINT_DIR / "bilstm_crf__supervised.pkl"

if bilstm_ckpt.exists():
    print("[SKIP] BiLSTM-CRF — checkpoint encontrado")
    with open(bilstm_ckpt, "rb") as f:
        cached = pickle.load(f)
    oof_true_bilstm = cached["oof_true"]
    oof_pred_bilstm = cached["oof_pred"]
else:
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    bio_data_arr = np.array(bio_data, dtype=object)

    oof_true_bilstm = [None] * len(bio_data)
    oof_pred_bilstm = [None] * len(bio_data)

    for fold_idx, (train_val_idx, test_idx) in enumerate(kf.split(bio_data_arr)):
        print(f'\n{"="*40} Fold {fold_idx+1}/{N_FOLDS} {"="*40}')
        
        np.random.seed(SEED + fold_idx)
        np.random.shuffle(train_val_idx)
        val_size = max(1, len(train_val_idx) // 9)
        val_idx = train_val_idx[:val_size]
        train_idx = train_val_idx[val_size:]
        
        train_data = bio_data_arr[train_idx].tolist()
        val_data = bio_data_arr[val_idx].tolist()
        test_data = bio_data_arr[test_idx].tolist()
        
        print(f'  Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')
        
        train_loader = DataLoader(NERDataset(train_data, word2id, label2id), batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(NERDataset(val_data, word2id, label2id), batch_size=8, shuffle=False, collate_fn=collate_fn)
        
        model = BiLSTM_CRF(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LABELS).to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_val_loss = float('inf')
        patience_counter = 0
        PATIENCE = 10
        EPOCHS = 50
        
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                loss = model(batch['input_ids'].to(device), batch['labels'].to(device), batch['mask'].to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                total_loss += loss.item()
            
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch in val_loader:
                    loss = model(batch['input_ids'].to(device), batch['labels'].to(device), batch['mask'].to(device))
                    val_loss += loss.item()
            
            avg_val = val_loss / len(val_loader)
            scheduler.step(avg_val)
            
            if avg_val < best_val_loss:
                best_val_loss = avg_val
                torch.save(model.state_dict(), f'best_bilstm_fold{fold_idx}.pt')
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break
        
        if (epoch + 1) == EPOCHS:
            print(f'  Completed {EPOCHS} epochs')
        
        model.load_state_dict(torch.load(f'best_bilstm_fold{fold_idx}.pt', map_location=device))
        model.eval()
        
        with torch.no_grad():
            for local_i, global_i in enumerate(test_idx):
                sample = bio_data[global_i]
                tokens = sample['tokens'][:512]
                true_labels = sample['labels'][:512]
                
                input_ids = torch.tensor(
                    [[word2id.get(t.lower(), word2id['<UNK>']) for t in tokens]],
                    dtype=torch.long
                ).to(device)
                mask = torch.ones_like(input_ids, dtype=torch.bool).to(device)
                pred_ids = model(input_ids, mask=mask)[0]
                pred_labels = [id2label[p] for p in pred_ids]
                
                oof_true_bilstm[global_i] = true_labels
                oof_pred_bilstm[global_i] = pred_labels
        
        del model, optimizer, scheduler
        gc.collect()
        if torch.backends.mps.is_available(): torch.mps.empty_cache()
        print(f'  Fold {fold_idx+1} done. OOF predictions: {sum(x is not None for x in oof_pred_bilstm)}/{len(bio_data)}')

    assert all(x is not None for x in oof_pred_bilstm), 'Alguns docs não receberam predição!'
    
    with open(bilstm_ckpt, "wb") as f:
        pickle.dump({"oof_true": oof_true_bilstm, "oof_pred": oof_pred_bilstm}, f)
    print(f"[SAVED] {bilstm_ckpt}")

print(f'\nBiLSTM-CRF: {len(oof_pred_bilstm)} documentos preditos via {N_FOLDS}-fold CV.')

Vocab size: 6007
BiLSTM-CRF definido: embed=128, hidden=256, labels=9

======================================== Fold 1/3 ========================================
  Train: 513, Val: 64, Test: 289


KeyboardInterrupt: 

In [ ]:
#bilstm_results = full_evaluation(bio_data, oof_true_bilstm, oof_pred_bilstm, 'BiLSTM-CRF')
#save_and_log_checkpoint('bilstm-crf', oof_true_bilstm, oof_pred_bilstm, bilstm_results, technique='supervised')


---
## 5. BERT Models — K-Fold

In [8]:
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback
)


class NERTokenDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        tokens = sample['tokens']
        labels = sample['labels']
        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            max_length=self.max_length, truncation=True, padding=False,
            return_offsets_mapping=False
        )
        word_ids = encoding.word_ids()
        aligned_labels = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                aligned_labels.append(self.label2id[labels[word_id]])
            else:
                lbl = labels[word_id]
                if lbl.startswith('B-'):
                    aligned_labels.append(self.label2id['I-' + lbl[2:]])
                else:
                    aligned_labels.append(self.label2id[lbl])
            previous_word_id = word_id
        encoding['labels'] = aligned_labels
        return {k: torch.tensor(v) for k, v in encoding.items()}


def compute_metrics_hf(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_labels_list, pred_labels_list = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_seq, pred_seq_clean = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_seq.append(id2label[l])
                pred_seq_clean.append(id2label[p])
        true_labels_list.append(true_seq)
        pred_labels_list.append(pred_seq_clean)
    return {
        'f1': seq_f1_score(true_labels_list, pred_labels_list, zero_division=0),
        'precision': seq_precision_score(true_labels_list, pred_labels_list, zero_division=0),
        'recall': seq_recall_score(true_labels_list, pred_labels_list, zero_division=0),
    }

print('BERT dataset e métricas definidos.')

BERT dataset e métricas definidos.


In [9]:
import ctypes
import gc

def force_cleanup():    
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
        torch.mps.synchronize()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [10]:
def train_bert_kfold(model_name, bio_data, label2id, id2label, n_folds=5,
                     epochs=15, batch_size=2, lr=3e-5):
    """
    Treina um modelo BERT com K-Fold CV e retorna OOF predictions para todos os docs.
    Salva checkpoint por fold e, ao final, consolida tudo em um único .pkl com DataFrame.

    CORREÇÃO: predições são colapsadas ao nível de PALAVRA (primeiro subtoken),
    para alinhar a granularidade com os tokens spaCy usados na avaliação dos LLMs.
    """
    print(f'\n{"#"*60}')
    print(f'  {model_name} — {n_folds}-Fold CV')
    print(f'{"#"*60}')

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    data_arr = np.array(bio_data, dtype=object)

    oof_true = [None] * len(bio_data)
    oof_pred = [None] * len(bio_data)

    safe_name = model_name.replace("/", "_")

    for fold_idx, (train_val_idx, test_idx) in enumerate(kf.split(data_arr)):
        print(f'\n--- Fold {fold_idx+1}/{n_folds} ---')

        # ── Checkpoint: pular fold já concluído ──
        fold_ckpt = Path(f'oof_fold{fold_idx}_{safe_name}.pkl')
        if fold_ckpt.exists():
            print(f'  Fold {fold_idx+1} já processado, carregando checkpoint.')
            saved = pickle.load(open(fold_ckpt, 'rb'))
            for gi, (t, p) in saved.items():
                oof_true[gi] = t
                oof_pred[gi] = p
            print(f'  OOF: {sum(x is not None for x in oof_pred)}/{len(bio_data)}')
            continue

        np.random.seed(SEED + fold_idx)
        np.random.shuffle(train_val_idx)
        val_size = max(1, len(train_val_idx) // 9)
        val_idx = train_val_idx[:val_size]
        train_idx = train_val_idx[val_size:]

        train_data = data_arr[train_idx].tolist()
        val_data = data_arr[val_idx].tolist()
        test_data_fold = data_arr[test_idx].tolist()

        print(f'  Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data_fold)}')

        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY,
            name=f'{model_name.split("/")[-1]}_fold{fold_idx+1}',
            config={'model': model_name, 'fold': fold_idx+1, 'n_folds': n_folds,
                    'lr': lr, 'epochs': epochs, 'seed': SEED},
            group=f'{model_name.split("/")[-1]}__kfold',
        )

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(label2id),
            id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True
        )

        train_ds = NERTokenDataset(train_data, tokenizer, label2id)
        val_ds = NERTokenDataset(val_data, tokenizer, label2id)
        test_ds = NERTokenDataset(test_data_fold, tokenizer, label2id)

        data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)
        output_dir = f'./results_{safe_name}_fold{fold_idx}'

        is_large_vocab = 'xlm' in model_name.lower() or 'roberta' in model_name.lower()
        effective_batch = 1 if is_large_vocab else batch_size
        accum_steps = 4 if is_large_vocab else max(1, 8 // batch_size)

        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy='epoch',
            save_strategy='epoch',
            learning_rate=lr,
            per_device_train_batch_size=effective_batch,
            gradient_accumulation_steps=accum_steps,
            num_train_epochs=epochs,
            weight_decay=0.01, warmup_ratio=0.1,
            load_best_model_at_end=True,
            metric_for_best_model='f1', greater_is_better=True,
            save_total_limit=1, logging_steps=10,
            fp16=torch.cuda.is_available(),
            gradient_checkpointing=True,
            optim='adafactor', max_grad_norm=1.0,
            dataloader_pin_memory=False, dataloader_num_workers=0,
            eval_accumulation_steps=1,
            report_to='wandb', seed=SEED,
            per_device_eval_batch_size=1,
        )

        trainer = Trainer(
            model=model, args=training_args,
            train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics_hf,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
        )

        trainer.train()

        # Predições no fold de teste
        predictions = trainer.predict(test_ds)
        preds = np.argmax(predictions.predictions, axis=2)
        labels_arr_pred = predictions.label_ids

        fold_save = {}
        for local_i, global_i in enumerate(test_idx):
            pred_seq = preds[local_i]
            label_seq = labels_arr_pred[local_i]

            # ── Recuperar word_ids para colapsar subtokens → palavras ──
            sample = test_data_fold[local_i]
            encoding = tokenizer(
                sample['tokens'],
                is_split_into_words=True,
                max_length=512,
                truncation=True,
                padding=False,
            )
            word_ids = encoding.word_ids()

            true_seq, pred_seq_clean = [], []
            prev_word_id = None
            for p, l, wid in zip(pred_seq, label_seq, word_ids):
                if l == -100:                # special tokens ([CLS], [SEP], [PAD])
                    prev_word_id = wid
                    continue
                if wid != prev_word_id:      # primeiro subtoken da palavra → manter
                    true_seq.append(id2label[l])
                    pred_seq_clean.append(id2label[p])
                # subtokens de continuação → ignorar
                prev_word_id = wid

            oof_true[global_i] = true_seq
            oof_pred[global_i] = pred_seq_clean
            fold_save[global_i] = (true_seq, pred_seq_clean)

        # ── Salvar checkpoint deste fold ──
        pickle.dump(fold_save, open(fold_ckpt, 'wb'))
        print(f'  Checkpoint salvo: {fold_ckpt}')

        wandb.finish()

        # Cleanup
        del model, trainer, train_ds, val_ds, test_ds, predictions, preds, labels_arr_pred
        force_cleanup()

        print(f'  Fold {fold_idx+1} done. OOF: {sum(x is not None for x in oof_pred)}/{len(bio_data)}')

    # ── Consolidar todos os folds em um único DataFrame ──
    assert all(x is not None for x in oof_pred), 'Docs sem predição!'

    df_oof = pd.DataFrame({
        'doc_idx': range(len(bio_data)),
        'true_labels': oof_true,
        'pred_labels': oof_pred,
        'model': model_name,
    })

    consolidated_path = Path(f'oof_consolidated_{safe_name}.pkl')
    df_oof.to_pickle(consolidated_path)
    print(f'\nDataFrame consolidado salvo: {consolidated_path}  ({len(df_oof)} registros)')

    # Limpar checkpoints parciais
    for f in Path('.').glob(f'oof_fold*_{safe_name}.pkl'):
        f.unlink()
    print('Checkpoints parciais removidos.')

    return oof_true, oof_pred

print('Função train_bert_kfold definida.')

Função train_bert_kfold definida.


In [ ]:
BERT_MODELS = [
    'neuralmind/bert-base-portuguese-cased',
    'neuralmind/bert-large-portuguese-cased',
    #'microsoft/mdeberta-v3-base',
    'rufimelo/Legal-BERTimbau-base',
    #PORTULAN/albertina-ptbr',
]

print(f'{len(BERT_MODELS)} modelos BERT a avaliar com {N_FOLDS}-fold CV.')

2 modelos BERT a avaliar com 3-fold CV.


In [12]:
#all_results = [bilstm_results]
all_results = []

for model_name in BERT_MODELS:
    ckpt_path = get_checkpoint_path(model_name, "supervised")
    
    if ckpt_path.exists():
        print(f"[SKIP] {model_name} / supervised — checkpoint encontrado")
        with open(ckpt_path, "rb") as f:
            cached = pickle.load(f)
        # Recalcular métricas com pipeline spaCy
        oof_true_cached = cached['true_labels']
        oof_pred_cached = cached['pred_labels']
        results = full_evaluation(bio_data, oof_true_cached, oof_pred_cached, model_name)
        all_results.append(results)
        continue
    
    try:
        oof_true, oof_pred = train_bert_kfold(
            model_name=model_name,
            bio_data=bio_data,
            label2id=label2id,
            id2label=id2label,
            n_folds=N_FOLDS,
            epochs=10,
            batch_size=1,
            lr=3e-5
        )
        results = full_evaluation(bio_data, oof_true, oof_pred, model_name)
        save_and_log_checkpoint(model_name, oof_true, oof_pred, results, technique='supervised')
        all_results.append(results)
    except Exception as e:
        print(f'ERRO com {model_name}: {e}')
        import traceback; traceback.print_exc()
    finally:
        force_cleanup() 

print(f'\n{len(all_results)} modelos avaliados com sucesso.')



############################################################
  microsoft/mdeberta-v3-base — 3-Fold CV
############################################################

--- Fold 1/3 ---
  Train: 513, Val: 64, Test: 289


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 42119.60it/s]
DebertaV2ForTokenClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.weight          

Epoch,Training Loss,Validation Loss


: 

In [ ]:
all_results

## 6. Final Comparison

In [ ]:
import os
import matplotlib.pyplot as plt

models = [r['model'] for r in all_results]
token_f1s = [r['token_f1'] for r in all_results]
span_f1s = [r['span_f1'] for r in all_results]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, token_f1s, width, label='F1 por Token', color='#2196F3')
bars2 = ax.bar(x + width/2, span_f1s, width, label='F1 por Span (IoU≥0.5)', color='#FF9800')

ax.set_ylabel('F1')
ax.set_title(f'Comparação de Modelos NER — decicontas.br ({N_FOLDS}-Fold CV, {len(bio_data)} docs)')
ax.set_xticks(x)
ax.set_xticklabels([m.split('/')[-1] for m in models], rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/ner_comparison_kfold.png', dpi=150)
plt.show()

In [ ]:
with open('ner_results_kfold.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print('Resultados salvos em ner_results_kfold.json')
print(f'Checkpoints salvos em {CHECKPOINT_DIR}')